In [1]:
import subprocess
import sys

# 필요한 패키지 리스트
required_packages = [
    "pypdf", 
    "langchain_community", 
    "langchain-openai", 
    "langchain-huggingface", 
    "transformers", 
    "faiss-gpu", 
    "langchain_anthropic", 
    "bitsandbytes"
]

# 패키지 설치 함수
def install_packages(packages):
    for package in packages:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package])
        except subprocess.CalledProcessError:
            print(f"Failed to install package: {package}")

# 패키지 설치 실행
install_packages(required_packages)


Failed to install package: faiss-gpu


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 모델 임베딩 설정
model_name = "jhgan/ko-sbert-nli"
hf_embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs={'device': 'cuda'})

# FAISS 인덱스 로드
faiss_index_path = "./faiss_index"  # FAISS 인덱스 경로
loaded_vector_store = FAISS.load_local(faiss_index_path, hf_embeddings)
retriever = loaded_vector_store.as_retriever(search_kwargs={'k': 5})


ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [ ]:
import requests

# 로컬 LM Studio 서버 설정
lm_studio_server_url = "http://172.30.1.44:1234/v1/chat/completions"

def query_lm_studio(prompt):
    headers = {"Content-Type": "application/json"}
    data = {
        "model": "EEVE-Korean-Instruct-10.8B-v1.0",  # 서버에 등록된 모델 이름
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 1024,
        "temperature": 0.7
    }
    response = requests.post(lm_studio_server_url, headers=headers, json=data)
    if response.status_code == 200:
        return response.json()["choices"][0]["message"]["content"]
    else:
        print(f"Error: {response.status_code}, {response.text}")
        return None


질문-응답 테스트를 합니다....

In [ ]:
from langchain.prompts import PromptTemplate

prompt_template = """
### [INST]
당신은 인플루언서를 추천하는 어시스턴스입니다. 주어진 정보를 기반으로 한국어로 성실하게 대답해주세요.
인플루언서 추천 시 인플루언서의 nickname을 알려줘야 합니다.
답변에 주어진 정보가 근거가 되어야 하고, 근거를 답변에 포함해 주세요.
주어진 정보로 질문에 답변하기 충분하지 않다면, "제가 가지고 있는 정보로는 답변이 어려울 것 같습니다."라고 대답해 주세요.
### 사용자의 질문:
{question}
### 주어진 정보:
{context}
[/INST]
"""
prompt = PromptTemplate(input_variables=["context", "question"], template=prompt_template)


In [ ]:
question_list = [
    "이 인플루언서는 caption과 해시태그에서 어떤 키워드를 가장 많이 사용했어?",
    "이 인플루언서는 어떤 캠페인을 제일 많이 했어?",
    "이 인플루언서는 게시글이나 릴스를 자주 올리는 사람이야?",
    "이 인플루언서는 게시물 수 대비 반응도가 높아?",
    "이 인플루언서는 팔로워 대비 게시물 반응도가 높아?",
    "이 인플루언서의 게시글 중 반응이 좋았던 게시글은 주로 어느 시간대에 포스팅된거야?",
    "이 인플루언서는 주로 어느 시간대에 게시글을 올려?",
    "이 인플루언서의 광고 게시글 중 반응이 좋았던 게시글의 키워드는 뭐가 있어?",
    "이 인플루언서는 biography를 보면 어떤 걸 알 수 있어? 예를 들어 유튜브 링크나 웹사이트 링크 등이 있을 수 있어.",
    "이 인플루언서의 팔로워 수는?",
    "이 인플루언서의 평균적인 댓글 수는?",
    "이 인플루언서 계정의 특징을 한 문장으로 요약해줘.",
    "이 인플루언서의 광고가 아닌 게시글 중 반응이 좋았던 게시글의 키워드는?",
    "이 인플루언서의 팔로워 성장세를 요약해줘.",
    "이 인플루언서는 릴스 광고에 적합한 사람일까?",
    "이 인플루언서는 게시글의 평균적인 캡션 길이가 길어?",
    "이 인플루언서는 게시글의 특징이 비슷해? 아니면 다양해?",
    "이 인플루언서는 게시글 중 광고와 광고가 아닌 게시글의 비율은?",
    "이 인플루언서가 광고를 잘 할 것 같은 제품을 몇 가지 알려줘.",
    "이 인플루언서의 게시글 중 가장 많은 좋아요를 받은 게시글의 내용 요약과 좋아요 개수를 알려줘.",
    "주로 어떤 계절이나 이벤트(예: 봄, 여름, 크리스마스 시즌)와 관련된 콘텐츠를 많이 올리나요?",
    "팔로워들은 이 인플루언서의 어떤 콘텐츠 형식(릴스 or 게시물)을 가장 좋아하나요?",
    "이 인플루언서가 최근 홍보한 제품이나 서비스는 무엇인가요?"
]

for question in question_list:
    # FAISS에서 정보 검색
    retrieved_docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in retrieved_docs])

    # 프롬프트 생성
    full_prompt = prompt.format(context=context, question=question)

    # LM Studio에 요청
    response = query_lm_studio(full_prompt)

    print("================================")
    print(f"질문: {question}")
    print("답변:")
    print(response)


In [7]:
import torch

print("PyTorch 버전:", torch.__version__)


ModuleNotFoundError: No module named 'torch'